# Lab 4: Exploitability, Not Head-to-Head

Two teams train agents for the same game and settle it with a tournament: every agent plays every other, and the
one with the best record is declared the strongest. Most people would ship the winner.

In this lab you write the parts of counterfactual regret minimization (CFR) from lecture 4.3, solve the lecture's
poker game with them, then run them on a game too large to inspect: phantom tic-tac-toe, where neither player sees
the other's marks. You hold a tournament among the strategies you train, and then you ask each strategy a harder
question: how much would an opponent who knew it exactly win against it?

| | |
|---|---|
| Runtime | a CPU is all it needs; the slowest cell takes a few minutes |
| Games | Kuhn poker (12 information sets) and phantom tic-tac-toe with a fixed opening (76,259) |
| Package | exp-a-spiel by Gabriele Farina, with the course's patch for the fixed opening |

> Save your own copy first: File → Save a copy in Drive.

## What is graded

- Checkpoint 1 (20%): your four functions on changed cases from the course page, and two values from Report values.
- Checkpoint 2 (30%): the strategy your issued run produces, uploaded and checked against the same run, and what a
  record of that run must say.
- Checkpoint 3 (35%): which claims an issued comparison supports.
- Checkpoint 4 (15%): what stays fixed in a fair next experiment.

Your strategy's exploitability and its tournament results are reported back to you. Neither is graded.

In [ ]:
#@title Install the lab's package and code (run this cell; there is no need to read it)
import hashlib, os, pathlib, subprocess, sys, urllib.request

RELEASE = "https://github.com/codey-m/optimization/releases/download/eas-v1/"
WHEELS = {'cp310': ('eas-0.0.1.post1-cp310-cp310-manylinux_2_28_x86_64.whl', '92ed71154c7b15c809a186c5a6a607fdfa19ba7f05f38d327e1f997fd991105a'), 'cp311': ('eas-0.0.1.post1-cp311-cp311-manylinux_2_28_x86_64.whl', '3196167e174fee94080e22232ef14973d031ea712485d92f3ce8b0df43e81278'), 'cp312': ('eas-0.0.1.post1-cp312-cp312-manylinux_2_28_x86_64.whl', 'b35579c95da2647f528f56ee1d74435ac47e173d7904a24f2c8f5278e905fb4d'), 'cp313': ('eas-0.0.1.post1-cp313-cp313-manylinux_2_28_x86_64.whl', 'd0c7fdaccdffea5619ef13f1e9d18949341e7d2d10b66b3433c8b9fb0d833636')}
CODE_BASE = os.environ.get("LAB4_BASE", "https://raw.githubusercontent.com/codey-m/optimization/main/labs/helpers/")
HELPERS_DIGEST = "06a0af793fbee9c538b9c6e246c8047732734b3a9957d3adcad61244db604235"

def _sha256(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()

try:
    import eas
    ready = hasattr(eas, "set_pttt_opening")
except ImportError:
    ready = False
if not ready:
    wheel = os.environ.get("EAS_WHEEL")
    if not wheel:
        tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
        if tag not in WHEELS:
            raise RuntimeError(f"There is no build of the lab's package for Python {sys.version.split()[0]} yet.")
        wheel, digest = WHEELS[tag]
        urllib.request.urlretrieve(RELEASE + wheel, wheel)
        if _sha256(wheel) != digest:
            raise RuntimeError(f"{wheel} did not download intact. Delete it and run this cell again.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)

if not (os.path.exists("lab4_helpers.py") and _sha256("lab4_helpers.py") == HELPERS_DIGEST):
    if CODE_BASE.startswith("http"):
        urllib.request.urlretrieve(CODE_BASE + "lab4_helpers.py", "lab4_helpers.py")
    else:
        pathlib.Path("lab4_helpers.py").write_bytes((pathlib.Path(CODE_BASE) / "lab4_helpers.py").read_bytes())
if _sha256("lab4_helpers.py") != HELPERS_DIGEST:
    raise RuntimeError("lab4_helpers.py did not download intact. Delete it and run this cell again.")
import lab4_helpers as H
print("Ready: the game package and the lab's provided code are installed.")

In [ ]:
import numpy as np

## Meet the game

Phantom tic-tac-toe is tic-tac-toe in the dark. On your turn you probe a cell. If it is empty your mark goes there;
if your opponent already holds it, the probe fails, you learn that, and the turn passes. You only ever see your own
marks and the cells you have found taken. Here the opening is fixed and known to both players: X holds the center
and O the top-left corner. Everything after that is played in the dark.

In [ ]:
H.show_game()

What a player has seen, the information-set string, is all a strategy can depend on. A strategy is a table: one row
per information set, nine columns for the nine cells, each row a probability distribution over the cells that
player may still probe.

## The layout every function uses

Your functions work on any game written this way, and you will run the same code on both games. Each player's
information sets are rows, ordered so that a parent comes before its children. `parent[i]` is the row of the
player's previous decision, or −1 at a first decision, and `parent_action[i]` is the action the player took there.
In Kuhn poker, player 0's rows 3 to 5 are the decisions after checking and then facing a bet, so their parent is the
first decision with the same card and their parent action is check (action 0).

In [ ]:
kuhn = H.Kuhn()
for i, key in enumerate(kuhn.keys[0]):
    print(f"player 0, row {i}: {key:16} parent {kuhn.parent[0][i]:>2}, parent action {kuhn.parent_action[0][i]:>2}")

## Your task 1: Regret matching

Each row is one decision point with its own regret minimizer, as in lecture 4.3's picture of CFR as a team of local
learners. `regret_matching(R, legal)` turns a table of accumulated regrets into the strategy the learners play next.

<details><summary>Hint</summary>

Each row is separate. Only regret that is positive counts, and only on legal actions; play each legal action in
proportion to its share of that positive regret. A row with nothing positive gives you no reason to prefer any
action, so every legal action gets the same probability. Illegal actions always get zero.

</details>

In [ ]:
def regret_matching(R, legal):
    """R: [rows, actions] accumulated regrets. legal: [rows, actions] booleans. Returns a strategy of the same shape."""
    # TODO: one distribution per row from the positive regrets on legal actions.
    raise NotImplementedError("Write regret matching.")

In [ ]:
toy = np.array([[1.0, 3.0, -2.0], [-1.0, -0.5, 0.0], [2.0, 0.0, 5.0]])
toy_legal = np.array([[True, True, True], [True, True, False], [True, True, False]])
s = regret_matching(toy, toy_legal)
assert s.shape == toy.shape, "Return one row per decision point, one column per action."
assert np.allclose(s.sum(1), 1), "Each row must sum to 1."
assert (s[~toy_legal] == 0).all(), "Illegal actions get zero probability."
assert np.allclose(s[0], [0.25, 0.75, 0.0]), "Row 0: play in proportion to the positive regrets."
assert np.allclose(s[1], [0.5, 0.5, 0.0]), "Row 1 has no positive regret: play its legal actions equally."
assert np.allclose(s[2], [1.0, 0.0, 0.0]), "Row 2: a positive regret on an illegal action must not count."
print("Task 1 checks pass.")

## Your task 2: Sequence form

A behavioral strategy says how to play at each decision point. Lecture 4.3's sequence form stores instead the
probability of the whole sequence of the player's own actions that leads to each entry, which is what makes
expected payoff linear in the strategy. `sequence_form(x, parent, parent_action)` converts one to the other.

<details><summary>Hint</summary>

Work down the tree, parents first, which is the order the rows are already in. A first decision keeps its
probabilities. Any other row's probabilities are scaled by the chance that the player reached it, and that chance
is the parent row's entry for the action that led here, already converted. Lecture 4.3's example multiplies 0.1 by
0.8 to get 0.08.

</details>

In [ ]:
def sequence_form(x, parent, parent_action):
    """x: [rows, actions] behavioral strategy, rows ordered parent first. Returns its sequence form, same shape."""
    # TODO: scale each row by the reach of the action that led to it.
    raise NotImplementedError("Write the sequence form.")

In [ ]:
x = np.array([[0.1, 0.9], [0.8, 0.2], [0.5, 0.5]])
sf = sequence_form(x, np.array([-1, 0, 1]), np.array([-1, 0, 1]))
assert np.allclose(sf[0], x[0]), "A first decision keeps its probabilities."
assert np.allclose(sf[1], [0.08, 0.02]), "Lecture 4.3's example: 0.1 times 0.8 is 0.08."
assert np.isclose(sf[2].sum(), sf[1, 1]), "A row's entries add up to the reach of the action that led to it."
rng = np.random.default_rng(0)
xr = rng.random((6, 2)); xr /= xr.sum(1, keepdims=True)
sk = sequence_form(xr, kuhn.parent[0], kuhn.parent_action[0])
assert np.allclose(sk[3:].sum(1), sk[:3, 0]), "In Kuhn poker, rows 3 to 5 together carry the reach of checking."
print("Task 2 checks pass.")

## Your task 3: Counterfactual values

The gradient gives each entry the payoff of the games that end right after that action. CFR's local learners need
more: the value of everything that can follow an action, with the player's own later choices played by the current
strategy. Lecture 4.3 builds these counterfactual values with a single pass from the bottom of the tree.
`counterfactual_values(g, x, parent, parent_action)` returns them, one per row and action.

<details><summary>Hint</summary>

Start from a copy of the gradient and work up the tree, children before parents, the reverse of the order the rows
are in. When a row is finished, its value is what the strategy expects to earn there: each action's counterfactual
value weighted by its probability. That value belongs to the action that led to the row, so add it to the parent's
entry for that action. Several rows can follow the same action (the player observed different things after it),
and each adds its own value.

</details>

In [ ]:
def counterfactual_values(g, x, parent, parent_action):
    """g: [rows, actions] gradient. x: [rows, actions] behavioral strategy. Returns counterfactual values, same shape."""
    # TODO: one pass from the bottom of the tree, passing each row's value up to the action that led to it.
    raise NotImplementedError("Write the counterfactual values.")

In [ ]:
g = np.array([[1.0, 0.0], [0.0, 2.0], [4.0, -2.0]])
cfv = counterfactual_values(g, x, np.array([-1, 0, 1]), np.array([-1, 0, 1]))
assert np.allclose(cfv[2], g[2]), "A row with nothing after it keeps its gradient."
expected = (sequence_form(x, np.array([-1, 0, 1]), np.array([-1, 0, 1])) * g).sum()
assert np.isclose(x[0] @ cfv[0], expected), ("The first decision's value must equal the expected payoff, reach times "
                                             "gradient summed over every entry (lecture 4.3: payoff is linear in the sequence form).")
x0, x1 = H.Kuhn.equilibrium(); g0, g1, ev0 = kuhn.gradient(x0, x1)
roots = kuhn.parent[0] < 0
assert np.isclose((x0[roots] * counterfactual_values(g0, x0, kuhn.parent[0], kuhn.parent_action[0])[roots]).sum(), ev0)
print("Task 3 checks pass.")

## Your task 4: Best response value

How good is a strategy? Ask what an opponent who knew it exactly could win. `best_response_value(g, parent,
parent_action, legal)` is that opponent's total, given the gradient the strategy creates for them. Adding both
players' best-response totals gives NashConv: zero exactly at an equilibrium, and the larger it is, the more an
informed opponent gains. Exploitability, as lecture 4.4 reports it, measures the same gap.

<details><summary>Hint</summary>

The same climb as task 3, with one change: a best responder does not mix. At each row it takes the best legal entry,
and that best entry is the value passed up to the action that led to the row. At the top, add up the first
decisions' best entries.

</details>

In [ ]:
def best_response_value(g, parent, parent_action, legal):
    """g: [rows, actions] gradient. legal: [rows, actions] booleans. Returns the best responder's total as a float."""
    # TODO: one pass from the bottom, taking the best legal entry at each row.
    raise NotImplementedError("Write the best response value.")

In [ ]:
assert abs(H.nash_conv(kuhn, H.Kuhn.equilibrium(), best_response_value)) < 1e-9, \
    "At Kuhn poker's equilibrium neither best responder gains anything."
assert abs(H.nash_conv(kuhn, kuhn.uniform_pair(), best_response_value) - 0.9166667) < 1e-6, \
    "Against two uniform players, the best responders gain 0.9167 together."
g0, g1, _ = kuhn.gradient(*kuhn.uniform_pair())
assert best_response_value(g0, kuhn.parent[0], kuhn.parent_action[0], kuhn.legal[0]) >= \
    (kuhn.uniform(0) * counterfactual_values(g0, kuhn.uniform(0), kuhn.parent[0], kuhn.parent_action[0]))[:3].sum()
print("Task 4 checks pass.")

## Solve the lecture's game

`H.run_cfr` is the loop that puts your functions together, as in lecture 4.3's Algorithm 1: compute the gradients,
turn them into counterfactual values, update each row's regrets, and let regret matching choose the next strategy.
It keeps a running average of the strategies in sequence form, using your task 2. A traversal is one computation of
both players' gradients; it is the unit of work, and every budget in this lab counts traversals.

Here it runs CFR+ for 1,000 traversals. Lecture 4.3 gives Kuhn poker's value for player 0: −1/18.

In [ ]:
run = H.run_cfr(kuhn, "C4", [1000], regret_matching, sequence_form, counterfactual_values)
average = run.average[1000]
print(f"player 0's expected payoff {kuhn.gradient(*average)[2]:.5f} (lecture: {-1/18:.5f}); "
      f"NashConv {H.nash_conv(kuhn, average, best_response_value):.5f}")

## Same code, bigger game

The phantom tic-tac-toe game has 46,824 information sets for X and 29,435 for O. The package computes its gradients;
everything else is your code. The next cell first checks your functions against the package's own: best responses
to the uniform strategy, and three traversals of plain CFR.

In [ ]:
game = H.CornerPTTT()
check = H.check_against_package(game, regret_matching, sequence_form, counterfactual_values, best_response_value)
assert check["best_response"] < 1e-9, f"Your best response differs from the package's by {check['best_response']:.2e}."
assert check["cfr"] < 1e-9, f"Your CFR differs from the package's by {check['cfr']:.2e} after three traversals."
print(f"{game.rows(0):,} rows for X and {game.rows(1):,} for O; your functions agree with the package.")

Plain CFR (C0) updates both players every traversal and averages every strategy equally. CFR changes its strategy
at every step; the average is what lecture 4.3's guarantee is about.

> **Predict** Over 100 traversals, does the current strategy settle down, or does only the average?

In [ ]:
PREDICT_SETTLES = None   # "both", "only the average", or "neither"

In [ ]:
assert PREDICT_SETTLES in ("both", "only the average", "neither"), 'Choose "both", "only the average" or "neither".'
runs = {"C0": H.run_cfr(game, "C0", [10, 30, 100], regret_matching, sequence_form, counterfactual_values)}
H.print_ladder(runs, game, best_response_value, current=True)
H.plot_runs(runs, game, best_response_value, "Plain CFR: average (solid) and current (dashed)")
print("You predicted:", PREDICT_SETTLES)

## One change at a time

Lecture 4.3 ends with CFR+, which changes three things at once: regret matching+ (negative regret is forgotten),
alternating updates (one player per traversal, each seeing the other's newest strategy), and linear averaging (later
strategies count more). Each run below changes one of them against plain CFR, and the last changes all three.

> **Predict** By 100 traversals, which single change helps most: "C1" regret matching+, "C2" alternation, or "C3"
> linear averaging?

In [ ]:
PREDICT_BEST = None   # "C1", "C2" or "C3"

In [ ]:
assert PREDICT_BEST in ("C1", "C2", "C3"), 'Choose "C1", "C2" or "C3".'
for c in ("C1", "C2", "C3", "C4"):
    runs[c] = H.run_cfr(game, c, [10, 30, 100], regret_matching, sequence_form, counterfactual_values)
H.print_ladder(runs, game, best_response_value)
print("You predicted:", PREDICT_BEST)

Alternation updates one player per traversal, so it makes half as many updates for the same work, and the table
counts work.

## The table everyone would read

Now the tournament. Every average strategy from the ladder, at 10, 30 and 100 traversals, plays every other once in
each seat, and the uniform strategy joins them. The head-to-head result is exact, the expected payoff per game, with
no sampling noise.

> **Predict** Will the strategy with the best tournament record also be the least exploitable?

In [ ]:
PREDICT_LEADER = None   # "yes" or "no"

In [ ]:
assert PREDICT_LEADER in ("yes", "no"), 'Choose "yes" or "no".'
pool = {f"{c}@{t}": run.average[t] for c, run in runs.items() for t in (10, 30, 100)}
pool["uniform"] = game.uniform_pair()
names, table, field = H.head_to_head_table(game, pool)
nash = {n: H.nash_conv(game, pool[n], best_response_value) for n in names}
H.print_head_to_head(names, field, nash)
print("You predicted:", PREDICT_LEADER)

A tournament scores a strategy against the opponents that happened to enter. NashConv scores it against the one
opponent that knows it best. Which of the two would you report for an agent about to meet players you have never
tested?

## Changed cases

Checkpoint 1 gives you new inputs for your four functions. Replace the four `CASE_` lines with the block from the
course page, run the cell, and enter the four values it prints. The small tree is the same for everyone: rows 3 and
4 both follow row 1's action 1, after the player observed different things.

In [ ]:
CASE_R = [[2.0, -1.0, 3.0, 1.5], [-0.5, -2.0, 1.0, -1.0]]
CASE_LEGAL = [[True, True, True, False], [True, True, False, True]]
CASE_X = [[0.3, 0.7], [0.6, 0.4], [0.25, 0.75], [0.5, 0.5], [0.9, 0.1]]
CASE_G = [[1.0, -2.0], [0.5, 1.5], [-1.0, 2.0], [3.0, -1.0], [2.0, 2.0]]

TREE_PARENT, TREE_ACTION = np.array([-1, 0, 0, 1, 1]), np.array([-1, 0, 1, 1, 1])
R, L, X, G = np.array(CASE_R), np.array(CASE_LEGAL), np.array(CASE_X), np.array(CASE_G)
C1 = regret_matching(R, L)[:, 0].sum()
C2 = sequence_form(X, TREE_PARENT, TREE_ACTION)[3, 0]
C3 = counterfactual_values(G, X, TREE_PARENT, TREE_ACTION)[0, 0]
C4 = best_response_value(G, TREE_PARENT, TREE_ACTION, np.ones(X.shape, bool))
print(f"C1 = {C1:.4f}\nC2 = {C2:.4f}\nC3 = {C3:.4f}\nC4 = {C4:.4f}")

## Report values

In [ ]:
print(f"D1 = {game.rows(0)}   (information sets for X)")
print(f"D2 = {H.nash_conv(game, game.uniform_pair(), best_response_value):.4f}   (NashConv of the uniform strategy pair)")

## Your run

Checkpoint 2 issues you a budget between 80 and 120 traversals. Run CFR+ for exactly that many and upload the
average strategy it produces. The course checks it against the same run, so a correct implementation passes and the
strategy's strength does not matter. The service also reports your strategy's NashConv and its head-to-head results
against the five ladder strategies at 100 traversals, the same numbers the cell below prints.

> **Predict** Among those five strategies and yours, where will yours rank by head-to-head record (1 is best)?

In [ ]:
MY_BUDGET = None        # your budget from the course page's Checkpoint 2
PREDICT_RANK = None     # 1 to 6

In [ ]:
assert isinstance(MY_BUDGET, int) and 80 <= MY_BUDGET <= 120, "Enter the budget from Checkpoint 2, 80 to 120."
assert PREDICT_RANK in range(1, 7), "Predict a rank from 1 to 6."
mine = H.run_cfr(game, "C4", [MY_BUDGET], regret_matching, sequence_form, counterfactual_values).average[MY_BUDGET]
H.certify(game, mine, MY_BUDGET)
field_pool = {f"{c}@100": runs[c].average[100] for c in runs}
field_pool["yours"] = mine
names, table, field = H.head_to_head_table(game, field_pool)
nash = {n: H.nash_conv(game, field_pool[n], best_response_value) for n in names}
H.print_head_to_head(names, field, nash)
print("You predicted rank", PREDICT_RANK)

Download `lab4_submission.zip` from the Files panel on the left and upload it on the course page's Checkpoint 2.

## The issued comparison

Checkpoint 3 asks about one comparison: linear averaging alone (C3) against CFR+ (C4) at a budget the course page
gives you. Replace the budget below with it and run the cell.

In [ ]:
CHECK_BUDGET = 10   # replace with the course page's budget for Checkpoint 3

pair = {c: H.run_cfr(game, c, [CHECK_BUDGET], regret_matching, sequence_form, counterfactual_values,
                     quiet=True).average[CHECK_BUDGET] for c in ("C3", "C4")}
print(f"At {CHECK_BUDGET} traversals:")
print(f"  C3 against C4, head-to-head: {H.head_to_head(game, pair['C3'], pair['C4']):+.4f} per game for C3")
for c in pair:
    print(f"  {c} NashConv: {H.nash_conv(game, pair[c], best_response_value):.4f}")

## Reflection

These are not graded. Take them to the forum.

1. Did your strategy's tournament rank and its NashConv rank agree? What would a tournament among everyone's
   uploads tell you about yours?
2. A team reports that its new agent wins 55% of its games against the old one. What would you ask for before
   believing the new agent is closer to equilibrium?
3. Why does the average strategy settle when the current one does not?
4. Lecture 4.4's policy-gradient methods submit their current policy, not an average. To compare one with CFR+
   fairly, what would you hold fixed, and in what unit would you count the work?

Checkpoint 4 asks about the last question. Where to look: the issued comparison above, and the ladder.